# Compare Two Runs (No Widgets)

**Instructions:**
1. Run the "Load data" cell once.
2. Edit the values in the "Set parameters" cell.
3. Run the "Plot comparison" cell to see the result.
4. To change parameters, edit the values in the "Set parameters" cell and run that cell again (or simply re‑run the "Plot comparison" cell after editing).

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. Load data
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Find all_results.csv
candidates = [
    'all_results.csv',
    os.path.join('..', 'all_results.csv'),
]
csv_path = None
for p in candidates:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError("all_results.csv not found. Place it next to this notebook or one level up.")

df = pd.read_csv(csv_path)
df['algorithm'] = df['pca'].map({True: 'PCA-DEA', False: 'UMAP-DEA'})

print(f"Loaded {len(df)} rows from {csv_path}")
print("Available parameter values:")
print(f"  Algorithm: {sorted(df['algorithm'].unique())}")
print(f"  N:         {sorted(df['N'].unique())}")
print(f"  n:         {sorted(df['n'].unique())}")
print(f"  RTS:       {sorted(df['rts'].unique())}")
print(f"  Gamma:     {sorted(df['gamma'].unique())}")
print(f"  Metrics:   MAE, Spearman, Pearson, Kendall, Proportion Efficient, Number Efficient")
print(f"  umap_n_neighbors & nr_simulations depend on your (algo, N, n) choices —")
print(f"  run the Plot cell to see valid options for your parameters")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Set parameters (edit these values)
# ═══════════════════════════════════════════════════════════

# ── Run A ──
RUN_A_ALGO              = 'UMAP-DEA'   # 'UMAP-DEA' or 'PCA-DEA'
RUN_A_N                 = 50           # from printed N values
RUN_A_n                 = 200          # from printed n values
RUN_A_RTS               = 'vrs'        # 'vrs' or 'crs'
RUN_A_GAMMA             = 0.5          # from printed gamma values
RUN_A_UMAP_N_NEIGHBORS  = 7            # UMAP n_neighbors (used for filtering, even for PCA-DEA)
RUN_A_NR_SIMULATIONS    = 100          # nr_simulations

# ── Run B ──
RUN_B_ALGO              = 'UMAP-DEA'
RUN_B_N                 = 50
RUN_B_n                 = 200
RUN_B_RTS               = 'vrs'
RUN_B_GAMMA             = 0.5
RUN_B_UMAP_N_NEIGHBORS  = 14            # UMAP n_neighbors (used for filtering, even for PCA-DEA)
RUN_B_NR_SIMULATIONS    = 100          # nr_simulations

# ── Shared ──
METRIC_NAME = 'Kendall'    # any from the list above
SHOW_STD    = True         # True or False

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. Plot comparison (just run this cell after setting parameters)
# ═══════════════════════════════════════════════════════════

# Metric mapping
METRICS = {
    'MAE':                  ('mae_mean', 'mae_std'),
    'Spearman':             ('spearmanr_mean', 'spearmanr_std'),
    'Pearson':              ('pearsonr_mean', 'pearsonr_std'),
    'Kendall':              ('kendalltau_mean', 'kendalltau_std'),
    'Proportion Efficient': ('prop_efficient_mean', 'prop_efficient_std'),
    'Number Efficient':     ('nr_efficient_mean', 'nr_efficient_std'),
}
mean_col, std_col = METRICS[METRIC_NAME]

# Dimension order
DIM_ORDER = ['log', 'sqrt', 'ten_percent', 'half', 'original']
DIM_LABELS = ['log(N)', '√N', '10%', 'N/2', 'N (original)']
COLOR_A = '#1f77b4'
COLOR_B = '#ff7f0e'

# ── Helper: show valid sub-parameters for a given base filter ──
def show_valid_options(label, df_sub):
    if len(df_sub) == 0:
        print(f'  {label} → No matching data for these base parameters!')
        return
    valid_k  = sorted(df_sub['umap_n_neighbors'].unique())
    valid_nr = sorted(df_sub['nr_simulations'].unique())
    print(f'  Valid umap_n_neighbors for {label}: {valid_k}')
    print(f'  Valid nr_simulations    for {label}: {valid_nr}')

# Partial filter (base params only) to show valid choices
mask_a_base = (
    (df['algorithm'] == RUN_A_ALGO) &
    (df['N'] == RUN_A_N) &
    (df['n'] == RUN_A_n) &
    (df['rts'] == RUN_A_RTS) &
    (df['gamma'] == RUN_A_GAMMA)
)
mask_b_base = (
    (df['algorithm'] == RUN_B_ALGO) &
    (df['N'] == RUN_B_N) &
    (df['n'] == RUN_B_n) &
    (df['rts'] == RUN_B_RTS) &
    (df['gamma'] == RUN_B_GAMMA)
)

print('─' * 50)
show_valid_options(f'Run A ({RUN_A_ALGO}, N={RUN_A_N}, n={RUN_A_n}, {RUN_A_RTS}, γ={RUN_A_GAMMA})', df[mask_a_base])
show_valid_options(f'Run B ({RUN_B_ALGO}, N={RUN_B_N}, n={RUN_B_n}, {RUN_B_RTS}, γ={RUN_B_GAMMA})', df[mask_b_base])
print('─' * 50)

# Check that the user's sub-parameter choices exist
mask_a_check = mask_a_base & (df['umap_n_neighbors'] == RUN_A_UMAP_N_NEIGHBORS) & (df['nr_simulations'] == RUN_A_NR_SIMULATIONS)
mask_b_check = mask_b_base & (df['umap_n_neighbors'] == RUN_B_UMAP_N_NEIGHBORS) & (df['nr_simulations'] == RUN_B_NR_SIMULATIONS)
if mask_a_check.sum() == 0:
    valid_k  = sorted(df[mask_a_base]['umap_n_neighbors'].unique())
    valid_nr = sorted(df[mask_a_base]['nr_simulations'].unique())
    print(f'❌ No data for Run A with umap_n_neighbors={RUN_A_UMAP_N_NEIGHBORS}, nr_simulations={RUN_A_NR_SIMULATIONS}')
    print(f'   Valid umap_n_neighbors: {valid_k}')
    print(f'   Valid nr_simulations:    {valid_nr}')
    raise SystemExit
if mask_b_check.sum() == 0:
    valid_k  = sorted(df[mask_b_base]['umap_n_neighbors'].unique())
    valid_nr = sorted(df[mask_b_base]['nr_simulations'].unique())
    print(f'❌ No data for Run B with umap_n_neighbors={RUN_B_UMAP_N_NEIGHBORS}, nr_simulations={RUN_B_NR_SIMULATIONS}')
    print(f'   Valid umap_n_neighbors: {valid_k}')
    print(f'   Valid nr_simulations:    {valid_nr}')
    raise SystemExit

# Full filter data
mask_a = mask_a_check
mask_b = mask_b_check

df_a = df[mask_a].copy()
df_b = df[mask_b].copy()

# Align levels
levels_present = set(df_a['dim_reduction_level'].unique()) & set(df_b['dim_reduction_level'].unique())
levels_ordered = [l for l in DIM_ORDER if l in levels_present]
labels_ordered = [DIM_LABELS[DIM_ORDER.index(l)] for l in levels_ordered]

if len(levels_ordered) == 0:
    print('⚠️ No common dim_reduction_level values between the two runs.')
    raise SystemExit

# Aggregate (mean over seeds)
agg_a = df_a.groupby('dim_reduction_level').agg(
    metric_mean=(mean_col, 'mean'),
    metric_std=(std_col, 'mean'),
    n_seeds=('seed', 'nunique'),
).reindex(levels_ordered).reset_index()

agg_b = df_b.groupby('dim_reduction_level').agg(
    metric_mean=(mean_col, 'mean'),
    metric_std=(std_col, 'mean'),
    n_seeds=('seed', 'nunique'),
).reindex(levels_ordered).reset_index()

# Plot
fig, (ax_bar, ax_diff) = plt.subplots(2, 1, figsize=(12, 8),
                                      gridspec_kw={'height_ratios': [3, 1.5]})
x = np.arange(len(levels_ordered))
width = 0.35

# Bars
yerr_a = agg_a['metric_std'].values if SHOW_STD else None
bars_a = ax_bar.bar(x - width/2, agg_a['metric_mean'].values, width,
                    yerr=yerr_a, capsize=5, color=COLOR_A, alpha=0.85, edgecolor='white',
                    label=f'Run A: {RUN_A_ALGO} (k={RUN_A_UMAP_N_NEIGHBORS})')
yerr_b = agg_b['metric_std'].values if SHOW_STD else None
bars_b = ax_bar.bar(x + width/2, agg_b['metric_mean'].values, width,
                    yerr=yerr_b, capsize=5, color=COLOR_B, alpha=0.85, edgecolor='white',
                    label=f'Run B: {RUN_B_ALGO} (k={RUN_B_UMAP_N_NEIGHBORS})')

# Annotate bars
for bar in bars_a:
    h = bar.get_height()
    if not np.isnan(h):
        ax_bar.text(bar.get_x() + bar.get_width()/2., h + 0.002,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=8, color=COLOR_A)
for bar in bars_b:
    h = bar.get_height()
    if not np.isnan(h):
        ax_bar.text(bar.get_x() + bar.get_width()/2., h + 0.002,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=8, color=COLOR_B)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(labels_ordered, fontsize=11)
ax_bar.set_ylabel(METRIC_NAME, fontsize=12)
ax_bar.set_title(
    f'{METRIC_NAME}: {RUN_A_ALGO} (N={RUN_A_N}, n={RUN_A_n}, k={RUN_A_UMAP_N_NEIGHBORS}, {RUN_A_RTS.upper()})  vs  '
    f'{RUN_B_ALGO} (N={RUN_B_N}, n={RUN_B_n}, k={RUN_B_UMAP_N_NEIGHBORS}, {RUN_B_RTS.upper()})',
    fontsize=14, fontweight='bold'
)
ax_bar.legend(fontsize=10, loc='best')
ax_bar.grid(True, alpha=0.3, axis='y')

# Difference panel
diff_values = agg_a['metric_mean'].values - agg_b['metric_mean'].values
diff_colors = [COLOR_A if d >= 0 else COLOR_B for d in diff_values]
ax_diff.bar(x, diff_values, width * 1.5, color=diff_colors, alpha=0.75, edgecolor='white')
ax_diff.axhline(y=0, color='black', linewidth=0.8)
for i, d in enumerate(diff_values):
    if not np.isnan(d):
        va = 'bottom' if d >= 0 else 'top'
        offset = 0.002 if d >= 0 else -0.002
        ax_diff.text(i, d + offset, f'{d:+.3f}', ha='center', va=va, fontsize=8,
                     color='black', fontweight='bold')
ax_diff.set_xticks(x)
ax_diff.set_xticklabels(labels_ordered, fontsize=11)
ax_diff.set_ylabel('Δ (A − B)', fontsize=12)
ax_diff.set_title('Difference (Run A − Run B)', fontsize=12, fontweight='bold', color='#555555')
ax_diff.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary table
table_data = {
    'Dim Reduction': labels_ordered,
    f'{RUN_A_ALGO} ({METRIC_NAME})': [f'{v:.5f}' if not np.isnan(v) else 'NaN' for v in agg_a['metric_mean'].values],
    f'{RUN_B_ALGO} ({METRIC_NAME})': [f'{v:.5f}' if not np.isnan(v) else 'NaN' for v in agg_b['metric_mean'].values],
    'Δ (A−B)': [f'{d:+.5f}' if not np.isnan(d) else 'NaN' for d in diff_values],
}
table_df = pd.DataFrame(table_data)
caption = (f"Comparison Table — {METRIC_NAME}   "
           f"(Run A: ns={RUN_A_NR_SIMULATIONS}, seeds={int(agg_a['n_seeds'].iloc[0])}   "
           f"Run B: ns={RUN_B_NR_SIMULATIONS}, seeds={int(agg_b['n_seeds'].iloc[0])})")
print(f'\n{caption}')
print('=' * len(caption))
display(table_df)